# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [1]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 250
SPATIAL_UNIT = "community" # options: census, hexa, community

In [2]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

## Preparations

In [3]:
# "Settings" / Decisions for the training data
if SPATIAL_UNIT == "census":    
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
elif SPATIAL_UNIT == "hexa":
    DATA_PATH_TRAIN = "../data/train_test_data/svm_hexa_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_hexa_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_hexa_test.parquet" 
elif SPATIAL_UNIT == "community": 
    DATA_PATH_TRAIN = "../data/train_test_data/svm_community_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_community_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_community_test.parquet" 
else:
    print("Warning: No type of Spatial Unit given, Used census tract")
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_demand",
    "date",
]

Load data and select features and target

In [4]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [5]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [6]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2024-01-11 07:00:00,1,4,7,0.0,1.000000,0.433884,-0.900969,0.965926,-2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,87.64,43.82,32.5,55.14,Unknown
1,2024-01-11 02:00:00,1,4,2,0.0,1.000000,0.433884,-0.900969,0.500000,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips
2,2024-01-08 11:00:00,1,1,11,0.0,1.000000,0.000000,1.000000,0.258819,-9.659258e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips
3,2024-02-20 06:00:00,2,2,6,0.5,0.866025,0.781831,0.623490,1.000000,6.123234e-17,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips
4,2024-02-16 00:00:00,2,5,0,0.5,0.866025,-0.433884,-0.900969,0.000000,1.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips


In [8]:
train_df = train_df.sample(n=5000, random_state=42)
train_df_grid = train_df.sample(n=500, random_state=42)

In [9]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)

y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'is_holiday', 'community_area', 'food_drink', 'landmark', 'shop', 'train_station']
Target: uint32


In [10]:
# Create X and y for grid search (same encoding + scaling as the full training set)
X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)
X_train_grid = scaler.transform(X_train_grid)

y_train_grid = train_df_grid[TARGET_COL]

In [12]:
model = SVR()

In [13]:
param_grid = {
    "C": [1, 10, 100],
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

# Perform grid search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,               
    scoring="r2",  
    n_jobs=-1,
    error_score="raise"
)

# Fit
grid_search.fit(X_train_grid, y_train_grid)


,estimator,SVR()
,param_grid,"{'C': [1, 10, ...], 'gamma': ['scale', 'auto', ...], 'kernel': ['linear', 'rbf', ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,'raise'
,return_train_score,False
,kernel,'poly'


In [14]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 1, 'gamma': 0.1, 'kernel': 'poly'}
Best CV score: 0.685878588166197


In [15]:
# Train SVR 

model.fit(X_train, y_train, kernel=grid_search.best_params_["kernel"])

TypeError: BaseLibSVM.fit() got an unexpected keyword argument 'kernel'

In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([0.86370242, 1.03050174, 1.12443349, ..., 0.58577163, 0.39647657,
       0.71869768], shape=(223531,))

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 7.9992880345730235
MSE: 1226.8952250191385
RMSE: 35.027064179276266
R2 Score: 0.0077389154144491545


In [ ]:
# save model
dump(best_model, "../models/svm_" + SPATIAL_UNIT + "_svr.joblib")

['../models/svmcommunity_svr.joblib']